# ✅ RESOLUCIÓN (docente): pedidos de una tienda online
**Dataset:** `pedidos_tienda_online.csv` · 300 filas · 10 columnas (julio–septiembre 2026).
**Tiempo:** ~10 minutos · **Entregable:** notebook ejecutado + decisiones justificadas.

> Recordá: *no limpies sin entender el contexto*. Lo importante es tu **criterio**, no solo el código.

## 1. Carga y vista inicial
Cargá `pedidos_tienda_online.csv`, mostrá las primeras filas, la forma (`shape`) y ejecutá `info()`.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('pedidos_tienda_online.csv')
print(df.shape)
df.head()

(300, 10)


,id_pedido,id_cliente,fecha_pedido,categoria,monto,edad,pais,calificacion,cupon,telefono_alternativo
0,P-00001,C-0190,2026-07-01,Moda,NaN,27.0,Argentina,NaN,NaN,NaN
1,P-00002,C-0081,2026-07-01,Electrónica,562100.0,41.0,Uruguay,NaN,BLACK20,NaN
2,P-00003,C-0129,2026-07-01,Electrónica,371800.0,52.0,Argentina,NaN,NaN,NaN
3,P-00004,C-0158,2026-07-01,Electrónica,608700.0,47.0,Argentina,5.0,BIENVENIDO10,+54 9 351 972-4165
4,P-00005,C-0124,2026-07-02,Hogar,140100.0,27.0,Argentina,5.0,NaN,NaN


## 2. Medición de faltantes
Armá una tabla con **cantidad** y **porcentaje** de nulos por columna, ordenada de mayor a menor.
⚠️ Dividí por el total de *filas*, no de columnas.

In [2]:
resumen = pd.DataFrame({
    'nulos': df.isna().sum(),
    'porcentaje': (df.isna().sum() / len(df) * 100).round(1)
}).sort_values('porcentaje', ascending=False)
resumen

,nulos,porcentaje
telefono_alternativo,253,84.3
cupon,176,58.7
calificacion,104,34.7
edad,26,8.7
monto,9,3.0
pais,6,2.0
id_cliente,0,0.0
id_pedido,0,0.0
fecha_pedido,0,0.0
categoria,0,0.0


## 3. Vacíos disfrazados 🕵️
Revisá `pais` con `value_counts(dropna=False)`. ¿Hay valores que *parecen* vacíos pero no son NaN?
Convertilos a NaN y volvé a medir.

In [3]:
print(df['pais'].value_counts(dropna=False))
df = df.replace(r'^\s*$', np.nan, regex=True)
df.isna().sum()

pais
Argentina    225
Chile         27
Uruguay       26
Paraguay      12
NaN            6
               4
Name: count, dtype: int64


id_pedido                 0
id_cliente                0
fecha_pedido              0
categoria                 0
monto                     9
edad                     26
pais                     10
calificacion            104
cupon                   176
telefono_alternativo    253
dtype: int64

## 4. Duplicados
a) ¿Cuántas filas están **completamente** duplicadas?
b) ¿Cuántos `id_pedido` se repiten?
c) ¿Cuántos `id_cliente` se repiten? **¿Son errores?** Justificá en una celda de texto.

In [4]:
print('Exactos       :', df.duplicated().sum())
print('Por id_pedido  :', df.duplicated(subset='id_pedido').sum())
print('Por id_cliente :', df.drop_duplicates().duplicated(subset='id_cliente').sum())
df[df.duplicated(keep=False)]

Exactos       : 8
Por id_pedido  : 8
Por id_cliente : 139


,id_pedido,id_cliente,fecha_pedido,categoria,monto,edad,pais,calificacion,cupon,telefono_alternativo
18,P-00019,C-0196,2026-07-08,Deportes,80300.0,60.0,Argentina,1.0,VERANO15,+54 9 351 995-1713
19,P-00019,C-0196,2026-07-08,Deportes,80300.0,60.0,Argentina,1.0,VERANO15,+54 9 351 995-1713
79,P-00079,C-0177,2026-07-26,Deportes,37100.0,40.0,Argentina,2.0,NaN,+54 9 351 795-8516
80,P-00079,C-0177,2026-07-26,Deportes,37100.0,40.0,Argentina,2.0,NaN,+54 9 351 795-8516
113,P-00112,C-0176,2026-08-03,Hogar,55400.0,39.0,Argentina,NaN,NaN,NaN
114,P-00112,C-0176,2026-08-03,Hogar,55400.0,39.0,Argentina,NaN,NaN,NaN
167,P-00165,C-0004,2026-08-18,Moda,118700.0,28.0,Argentina,5.0,NaN,NaN
168,P-00165,C-0004,2026-08-18,Moda,118700.0,28.0,Argentina,5.0,NaN,NaN
177,P-00174,C-0006,2026-08-23,Deportes,68800.0,18.0,Argentina,NaN,BIENVENIDO10,+54 9 351 645-2057
178,P-00174,C-0006,2026-08-23,Deportes,68800.0,18.0,Argentina,NaN,BIENVENIDO10,+54 9 351 645-2057


**Respuesta (c):** los `id_cliente` repetidos **no son errores**: un cliente puede hacer varios pedidos (cada uno con su `id_pedido` único). Los duplicados reales son los **8** con mismo `id_pedido` y todas las columnas idénticas (doble envío del formulario).

## 5. Análisis de impacto: decisiones
Para **cada columna con nulos**, completá en una celda de texto: *qué significa que falte*, *qué decisión tomás* y *por qué*.
Opciones: eliminar filas · imputar (media / mediana / moda / constante) · conservar los NaN · eliminar la columna.

| Columna | % | ¿Qué significa? | Decisión | Por qué |
|---|---|---|---|---|
| `telefono_alternativo` | 84,3 % | Dato opcional que casi nadie carga | **Eliminar columna** | >50 %: más ruido que información |
| `cupon` | 58,7 % | Nulo = no usó cupón (no es error) | **Rellenar `'SIN_CUPON'`** | Conserva información de negocio |
| `calificacion` | 34,7 % | No todos califican la compra | **Conservar NaN** (+ columna `tiene_calificacion`) | Imputar inventaría opiniones |
| `edad` | 8,7 % | Dato no capturado | **Imputar con la mediana (36)** | Variable útil; mediana robusta |
| `pais` (+4 espacios) | 3,3 % | Captura incompleta | **Imputar con la moda** | Categórica, ~75 % es Argentina |
| `monto` | 3,0 % | Fallo de registro | **Eliminar filas** | <5 % y es la variable principal: no se inventa |

## 6. Aplicar la limpieza
Implementá tus decisiones sobre una copia (`df_limpio`). Orden sugerido: duplicados → columna → filas → imputaciones.

In [5]:
df_limpio = df.drop_duplicates(subset='id_pedido').copy()

df_limpio = df_limpio.drop(columns='telefono_alternativo')
df_limpio = df_limpio.dropna(subset=['monto'])

df_limpio['cupon'] = df_limpio['cupon'].fillna('SIN_CUPON')
df_limpio['tiene_calificacion'] = df_limpio['calificacion'].notna()
df_limpio['edad'] = df_limpio['edad'].fillna(df_limpio['edad'].median())
df_limpio['pais'] = df_limpio['pais'].fillna(df_limpio['pais'].mode()[0])
df_limpio.shape

(283, 10)

## 7. Verificación y reporte
Mostrá cuántos nulos quedan, el `shape` final y escribí un reporte de 4–5 líneas para un supervisor que **no sabe programar**: qué encontraste y qué decidiste.

In [6]:
print('Filas: ', len(df), '->', len(df_limpio))
print(df_limpio.isna().sum())

Filas:  300 -> 283
id_pedido              0
id_cliente             0
fecha_pedido           0
categoria              0
monto                  0
edad                   0
pais                   0
calificacion          99
cupon                  0
tiene_calificacion     0
dtype: int64


**Reporte (modelo):** el archivo tenía 300 pedidos; 8 estaban duplicados por doble envío y se eliminaron. Se descartó la columna de teléfono alternativo (84 % vacía). Los 9 pedidos sin monto se quitaron (3 %). La edad faltante (9 %) se completó con la mediana y el país con el más frecuente. Los cupones vacíos significan *sin cupón* y la calificación faltante se dejó vacía porque no todos los clientes califican. Resultado: **283 pedidos** listos para analizar.

## 🎯 Desafío extra (opcional)
El promedio de `monto` está muy por encima de la mediana. ¿Por qué? Calculá ambos y, usando `groupby('categoria')`, mostrá la **mediana por categoría**.

In [7]:
print('Media  :', round(df_limpio['monto'].mean()))
print('Mediana:', df_limpio['monto'].median())
df_limpio.groupby('categoria')['monto'].median().sort_values(ascending=False)

Media  : 128976
Mediana: 65100.0


categoria
Electrónica    353600.0
Hogar           83700.0
Deportes        55600.0
Moda            44250.0
Libros          18900.0
Name: monto, dtype: float64

**Idea:** unos pocos pedidos de *Electrónica* (muy caros) elevan la media; la mediana representa mejor el pedido típico.